In [ ]:
# Fase 2 - PySpark, Puerto Coquimbo sitio 3
# Hito 1.1: Parametros, rutas y valores esperados del dominio
import glob
import os
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType, IntegerType, StringType, StructField, StructType,
)

RUTA_PROYECTO = "/opt/workspace"

# subcarpeta de la fase 1 que se procesa (w01, w02, w04, w08 o w16)
CONFIG_FASE1 = "w16"

RUTA_RAW = f"{RUTA_PROYECTO}/data/raw/{CONFIG_FASE1}"
PATRON_RAW = f"{RUTA_RAW}/*.jsonl"
RUTA_SALIDA = f"{RUTA_PROYECTO}/salida_fase2"   # en .gitignore, se usa en el hito 5

# valores esperados del dominio, salen de src/simulador.py (_inventario)
EQUIPOS_ESPERADOS = 69
PUNTOS_ESPERADOS = 185
METRICAS_ESPERADAS = {"vibration", "current", "weight", "temperature", "humidity"}
EVENTOS_ESPERADOS = 10_000_000   # src/main.py, TOTAL_EVENTOS

In [ ]:
# Hito 1.2: Comprobacion previa de los .jsonl desde el driver
archivos = sorted(glob.glob(PATRON_RAW))
if not archivos:
    raise FileNotFoundError(
        f"No hay .jsonl en {RUTA_RAW}. Revisa CONFIG_FASE1 o genera los datos "
        "con la fase 1 (python src/main.py)."
    )

total_bytes = sum(os.path.getsize(a) for a in archivos)
print(f"Archivos : {len(archivos)}")
print(f"Volumen  : {total_bytes / 1024**3:.2f} GiB")
for a in archivos[:5]:
    print(f"  {os.path.basename(a):<16} {os.path.getsize(a) / 1024**2:8.1f} MiB")
if len(archivos) > 5:
    print(f"  ... y {len(archivos) - 5} mas")

In [ ]:
# Hito 1.3: Sesion Spark contra el cluster y espera de executors
spark = (
    SparkSession.builder
    .appName("fase2-puerto-coquimbo")
    .config("spark.executor.memory", os.environ.get("SPARK_EXECUTOR_MEMORY", "1g"))
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("WARN")

# los executors tardan unos segundos en registrarse
esperados = int(os.environ.get("SPARK_WORKERS", "1"))
for _ in range(30):
    ejecutores = sc._jsc.sc().getExecutorMemoryStatus().size() - 1   # -1 = el driver
    if ejecutores >= esperados:
        break
    time.sleep(1)

print(f"Spark            : {spark.version}")
print(f"Master           : {sc.master}")
print(f"Executors vivos  : {ejecutores} (esperados {esperados})")
print(f"Paralelismo      : {sc.defaultParallelism}")
print(f"Zona horaria     : {spark.conf.get('spark.sql.session.timeZone')}")
print(f"AQE              : {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"maxPartitionBytes: {spark.conf.get('spark.sql.files.maxPartitionBytes')}")
if ejecutores == 0:
    raise RuntimeError("Sin executors: revisa SPARK_EXECUTOR_MEMORY <= SPARK_WORKER_MEMORY")

In [ ]:
# Hito 1.4: Esquema explicito (timestamp como texto, _corrupt_record para el diagnostico)
ESQUEMA_EVENTO = StructType([
    StructField("timestamp",       StringType(),  True),
    StructField("sensor_id",       StringType(),  True),
    StructField("metric",          StringType(),  True),
    StructField("value",           DoubleType(),  True),
    StructField("unit",            StringType(),  True),
    StructField("worker_id",       IntegerType(), True),
    StructField("_corrupt_record", StringType(),  True),
])

In [ ]:
# Hito 1.5: Lectura PERMISSIVE con columna de corruptos (perezosa)
df_raw = (
    spark.read
    .schema(ESQUEMA_EVENTO)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .json(PATRON_RAW)
)

df_raw.printSchema()
print(f"Particiones de lectura: {df_raw.rdd.getNumPartitions()}")

In [ ]:
# Hito 1.6: Muestra de la ingesta
df_raw.show(5, truncate=False)

In [ ]:
# Hito 1.6: Cache y conteo total, primera accion sobre todo el dataset
df_raw = df_raw.cache()

t0 = time.perf_counter()
total = df_raw.count()
t_lectura = time.perf_counter() - t0

print(f"Registros leidos : {total:,}")
print(f"Esperados        : {EVENTOS_ESPERADOS:,}")
print(f"Tiempo (lectura + cache): {t_lectura:.1f} s")
if total != EVENTOS_ESPERADOS:
    print("AVISO: el conteo no coincide con TOTAL_EVENTOS de la fase 1")

In [ ]:
# Hito 2: Perfilado y diagnostico de calidad (solo mide, no corrige)
# paso del reloj simulado de la fase 1 (src/generador.py, DT_MS = 15_000)
DT_S = 15

# unidad que la fase 1 le asigna a cada metrica (src/simulador.py)
UNIDAD_POR_METRICA = {
    "vibration": "mm/s", "current": "A", "weight": "kg",
    "temperature": "C", "humidity": "%",
}

# formato exacto que escribe la fase 1; se valida como texto, el parseo es del hito 3
PATRON_TS = r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d{3}Z$"

diagnostico = {}   # chequeo -> (valor medido, esperado, ok)
tiempos_h2 = {}    # seccion -> segundos de pared


def registrar(chequeo, medido, esperado, ok=None):
    ok = (medido == esperado) if ok is None else ok
    diagnostico[chequeo] = (medido, esperado, ok)
    print(f"[{'OK' if ok else 'FALLA'}] {chequeo}: {medido} (esperado {esperado})")

In [ ]:
# Hito 2.1: Estado del cache de df_raw
for info in sc._jsc.sc().getRDDStorageInfo():
    print(f"RDD          : {info.name()[:60]}")
    print(f"Particiones  : {info.numCachedPartitions()} de {info.numPartitions()} cacheadas")
    print(f"En memoria   : {info.memSize() / 1024**2:,.1f} MiB")
    print(f"En disco     : {info.diskSize() / 1024**2:,.1f} MiB")

In [ ]:
# Hito 2.2: Validez por linea en un solo agg (corruptas, nulos, formato, rangos)
t0 = time.perf_counter()

columnas = ["timestamp", "sensor_id", "metric", "value", "unit", "worker_id"]
fuera_de_rango = (
    (F.col("metric").isin("current", "vibration", "weight") & (F.col("value") < 0))
    | ((F.col("metric") == "humidity") & ~F.col("value").between(0, 100))
)

fila = df_raw.agg(
    F.count(F.col("_corrupt_record")).alias("corruptas"),
    *[F.sum(F.col(c).isNull().cast("int")).alias(f"nulos_{c}") for c in columnas],
    F.sum((~F.col("timestamp").rlike(PATRON_TS)).cast("int")).alias("ts_mal_formado"),
    F.sum(F.isnan("value").cast("int")).alias("value_nan"),
    F.sum(fuera_de_rango.cast("int")).alias("fuera_de_rango_fisico"),
).first().asDict()

tiempos_h2["2.2 validez por linea"] = time.perf_counter() - t0

registrar("lineas corruptas", fila["corruptas"], 0)
for c in columnas:
    registrar(f"nulos en {c}", fila[f"nulos_{c}"], 0)
registrar("timestamp mal formado", fila["ts_mal_formado"], 0)
registrar("value NaN", fila["value_nan"], 0)
registrar("value fisicamente imposible", fila["fuera_de_rango_fisico"], 0)
print(f"\nTiempo: {tiempos_h2['2.2 validez por linea']:.1f} s")

In [ ]:
# Hito 2.2: Muestra de lineas corruptas, si las hay
if fila["corruptas"]:
    df_raw.filter(F.col("_corrupt_record").isNotNull()) \
          .select("_corrupt_record").show(5, truncate=False)

In [ ]:
# Hito 2.3: Dominio: equipos, series, metricas y unidades
t0 = time.perf_counter()

dominio = df_raw.agg(
    F.countDistinct("sensor_id").alias("equipos"),
    F.countDistinct("sensor_id", "metric").alias("series"),
    F.collect_set("metric").alias("metricas"),
).first()

unidades = (
    df_raw.groupBy("metric")
    .agg(F.collect_set("unit").alias("unidades"), F.count("*").alias("lecturas"))
    .orderBy("metric")
    .collect()
)

tiempos_h2["2.3 dominio"] = time.perf_counter() - t0

registrar("equipos distintos", dominio["equipos"], EQUIPOS_ESPERADOS)
registrar("series (sensor_id, metric)", dominio["series"], PUNTOS_ESPERADOS)
registrar("metricas", sorted(dominio["metricas"]), sorted(METRICAS_ESPERADAS))

print()
for r in unidades:
    esperada = UNIDAD_POR_METRICA.get(r["metric"])
    registrar(f"unidad de {r['metric']}", sorted(r["unidades"]), [esperada])
    print(f"      lecturas: {r['lecturas']:,}")
print(f"\nTiempo: {tiempos_h2['2.3 dominio']:.1f} s")

In [ ]:
# Hito 2.4: Duplicados por clave y reparto de equipos por worker
t0 = time.perf_counter()

clave = ["sensor_id", "metric", "timestamp"]
duplicados = (
    df_raw.groupBy(clave).count()
    .filter(F.col("count") > 1)
    .agg(F.count("*").alias("claves"), F.sum(F.col("count") - 1).alias("filas_sobrantes"))
    .first()
)

workers_por_equipo = (
    df_raw.groupBy("sensor_id")
    .agg(F.countDistinct("worker_id").alias("n"))
    .filter(F.col("n") > 1)
    .count()
)

por_worker = df_raw.groupBy("worker_id").count().orderBy("worker_id").collect()

tiempos_h2["2.4 duplicados y reparto"] = time.perf_counter() - t0

registrar("claves duplicadas (sensor_id, metric, timestamp)", duplicados["claves"], 0)
registrar("filas sobrantes por duplicado", duplicados["filas_sobrantes"] or 0, 0)
registrar("equipos simulados por mas de un worker", workers_por_equipo, 0)
registrar("workers distintos", len(por_worker), len(archivos))

conteos = [r["count"] for r in por_worker]
print(f"\nEventos por worker: min {min(conteos):,} / max {max(conteos):,} "
      f"(max/min = {max(conteos) / min(conteos):.2f})")
print("El desbalance es esperado: los equipos tienen 1, 2 o 3 puntos de medicion.")
print(f"Tiempo: {tiempos_h2['2.4 duplicados y reparto']:.1f} s")

In [ ]:
# Hito 2.5: Continuidad de las series (huecos a 15 s)
t0 = time.perf_counter()

ticks_esperados = -(-EVENTOS_ESPERADOS // PUNTOS_ESPERADOS)   # division con techo

ts = F.to_timestamp("timestamp", "yyyy-MM-dd'T'HH:mm:ss.SSSX")
series = (
    df_raw.groupBy("sensor_id", "metric")
    .agg(
        F.count("*").alias("lecturas"),
        F.sum(ts.isNull().cast("int")).alias("ts_invalidos"),
        F.min(ts).alias("desde"),
        F.max(ts).alias("hasta"),
    )
    .withColumn(
        "huecos",
        ((F.unix_timestamp("hasta") - F.unix_timestamp("desde")) / DT_S + 1 - F.col("lecturas"))
        .cast("long"),
    )
    .cache()
)

resumen_series = series.agg(
    F.min("lecturas").alias("min_lecturas"),
    F.max("lecturas").alias("max_lecturas"),
    F.sum("huecos").alias("huecos"),
    F.sum((F.col("huecos") < 0).cast("int")).alias("series_con_exceso"),
    F.sum("ts_invalidos").alias("ts_invalidos"),
    F.min("desde").alias("desde"),
    F.max("hasta").alias("hasta"),
).first()

tiempos_h2["2.5 continuidad"] = time.perf_counter() - t0

registrar("lecturas minimas por serie", resumen_series["min_lecturas"], ticks_esperados - 1,
          ok=resumen_series["min_lecturas"] >= ticks_esperados - 1)
registrar("lecturas maximas por serie", resumen_series["max_lecturas"], ticks_esperados)
registrar("huecos en las series", resumen_series["huecos"], 0)
registrar("series con mas lecturas que ticks", resumen_series["series_con_exceso"], 0)
registrar("timestamp con fecha imposible", resumen_series["ts_invalidos"], 0)

dias = (resumen_series["hasta"] - resumen_series["desde"]).total_seconds() / 86400
print(f"\nPeriodo simulado: {resumen_series['desde']} -> {resumen_series['hasta']} ({dias:.2f} dias)")
print(f"Ticks esperados : {ticks_esperados:,} de {DT_S} s")
print(f"Tiempo: {tiempos_h2['2.5 continuidad']:.1f} s")

In [ ]:
# Hito 2.5: Series que alcanzan a emitir en el ultimo tick (cortado)
series.filter(F.col("lecturas") == ticks_esperados) \
      .select("sensor_id", "metric", "lecturas", "huecos") \
      .orderBy("sensor_id", "metric").show(truncate=False)
series.unpersist()

In [ ]:
# Hito 2.6: Perfil estadistico de value por metrica
t0 = time.perf_counter()

perfil = (
    df_raw.groupBy("metric")
    .agg(
        F.count("value").alias("n"),
        F.round(F.mean("value"), 3).alias("media"),
        F.round(F.stddev("value"), 3).alias("desv"),
        F.min("value").alias("min"),
        F.percentile_approx("value", [0.01, 0.5, 0.99]).alias("p01_p50_p99"),
        F.max("value").alias("max"),
    )
    .orderBy("metric")
)
perfil.show(truncate=False)

tiempos_h2["2.6 perfil de value"] = time.perf_counter() - t0
print(f"Tiempo: {tiempos_h2['2.6 perfil de value']:.1f} s")

In [ ]:
# Hito 2.6: Temperatura por familia (reefers vs ambientales)
(
    df_raw.filter(F.col("metric") == "temperature")
    .withColumn("familia", F.split("sensor_id", "_").getItem(0))
    .groupBy("familia")
    .agg(
        F.count("*").alias("n"),
        F.min("value").alias("min"),
        F.percentile_approx("value", [0.01, 0.5, 0.99]).alias("p01_p50_p99"),
        F.max("value").alias("max"),
    )
    .orderBy("familia")
    .show(truncate=False)
)

In [ ]:
# Hito 2.7: Resumen del diagnostico y tiempos
print(f"{'Chequeo':<52} {'Medido':>14} {'Esperado':>14}  Estado")
print("-" * 92)
for chequeo, (medido, esperado, ok) in diagnostico.items():
    print(f"{chequeo:<52} {str(medido):>14.14} {str(esperado):>14.14}  {'OK' if ok else 'FALLA'}")

fallas = [c for c, (_, _, ok) in diagnostico.items() if not ok]
print(f"\nChequeos: {len(diagnostico)}  |  OK: {len(diagnostico) - len(fallas)}  |  FALLA: {len(fallas)}")

print("\nTiempos del hito 2:")
for seccion, s in tiempos_h2.items():
    print(f"  {seccion:<28} {s:6.1f} s")
print(f"  {'total':<28} {sum(tiempos_h2.values()):6.1f} s")